# Planificar de verdad: el mundo de bloques

**Facsímil 2 · Inteligencia clásica** — capítulos 9 y 10 (planificación automática, PDDL y modelado
de dominios).

Hay una diferencia enorme entre *ejecutar* una secuencia de pasos y *deducir* cuál debe ser esa
secuencia. Lo segundo es **planificar**, y es lo que hace una IA cuando tiene que actuar en el mundo:
mover archivos, reservar recursos, operar una máquina, o —hoy— cuando un agente LLM «planea pasos».
En este cuaderno le das a un planificador un montón de bloques desordenados y una foto de cómo los
quieres, **sin decirle los pasos**. Solo describes las acciones posibles con sus precondiciones y
efectos, y él arma el plan. Esa separación entre «qué se puede hacer» y «cómo encadenarlo» es la idea
central de PDDL, el lenguaje estándar de la planificación automática.

### Qué vas a aprender
- Cómo se representa un problema de planificación: **estados** (conjuntos de hechos), **acciones**
  (precondiciones + efectos) y una **meta**.
- Que planificar es, otra vez, **buscar en un espacio de estados** (¡como el primer cuaderno!), pero
  ahora los estados son situaciones del mundo.
- Cómo un planificador **deduce** el orden correcto sin que se lo programes.
- Cómo el espacio de estados **explota** al añadir objetos, y por qué la planificación real necesita
  heurísticas.

### Cuánto cuesta
Unos 10 minutos. CPU, solo Python estándar.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
from collections import deque
import time

# Un estado es un conjunto de hechos verdaderos (predicados).
# Bloques A, B, C. Predicados: ('sobre',x,y) ('mesa',x) ('libre',x)
#                              ('mano_vacia',) ('cogido',x)
inicio = frozenset({('sobre','C','A'), ('mesa','A'), ('mesa','B'),
                    ('libre','C'), ('libre','B'), ('mano_vacia',)})
meta = {('sobre','A','B'), ('sobre','B','C')}   # torre A encima de B encima de C

def dibuja(estado):
    encima = {h[2]: h[1] for h in estado if h[0] == 'sobre'}
    bases = [h[1] for h in estado if h[0] == 'mesa']
    cols = []
    for base in sorted(bases):
        torre = [base]
        while torre[-1] in encima: torre.append(encima[torre[-1]])
        cols.append(torre)
    print("  ".join("/".join(reversed(c)) for c in cols), " (sobre la mesa)")
print("ESTADO INICIAL:"); dibuja(inicio)
print("META: torre A/B/C (A arriba del todo)")


ESTADO INICIAL:
C/A  B  (sobre la mesa)
META: torre A/B/C (A arriba del todo)


## 1. El estado: una foto del mundo como conjunto de hechos

En planificación clásica (el enfoque **STRIPS**, base de PDDL), un estado no es una imagen ni una
estructura complicada: es simplemente el **conjunto de hechos que son verdaderos** en ese momento. Por
ejemplo, «C está sobre A», «A está sobre la mesa», «la mano está vacía». Todo lo que no está en el
conjunto se asume falso. Esta representación tan austera es lo que hace la planificación tratable: un
estado es un conjunto, y comparar o transformar conjuntos es barato.


## 2. Las acciones: precondiciones y efectos

Aquí está el corazón de PDDL. Cada acción se describe con tres cosas:

- **Precondiciones:** qué hechos deben ser verdaderos para poder ejecutarla.
- **Efectos de borrado:** qué hechos dejan de ser verdaderos al ejecutarla.
- **Efectos de adición:** qué hechos pasan a ser verdaderos.

Por ejemplo, *coger un bloque de la mesa* requiere que esté libre, sobre la mesa y la mano vacía;
borra esos hechos y añade «cogido». **No hay ninguna instrucción sobre el orden**: el planificador lo
descubrirá. Definimos las cuatro acciones del mundo de bloques.


In [2]:
def acciones(estado):
    hechos = estado
    libres = {h[1] for h in hechos if h[0] == 'libre'}
    mano_vacia = ('mano_vacia',) in hechos
    cogido = next((h[1] for h in hechos if h[0] == 'cogido'), None)
    bloques = {h[1] for h in hechos if h[0] in ('mesa','libre','cogido')}
    res = []
    if mano_vacia:
        for x in libres:
            if ('mesa', x) in hechos:                       # COGER de la mesa
                nuevo = set(hechos) - {('mesa',x),('libre',x),('mano_vacia',)} | {('cogido',x)}
                res.append((f"coger {x} de la mesa", frozenset(nuevo)))
            for y in bloques:                               # DESAPILAR x de y
                if ('sobre',x,y) in hechos:
                    nuevo = set(hechos) - {('sobre',x,y),('libre',x),('mano_vacia',)} | {('cogido',x),('libre',y)}
                    res.append((f"desapilar {x} de {y}", frozenset(nuevo)))
    if cogido:
        nuevo = set(hechos) - {('cogido',cogido)} | {('mesa',cogido),('libre',cogido),('mano_vacia',)}
        res.append((f"dejar {cogido} en la mesa", frozenset(nuevo)))   # DEJAR en la mesa
        for y in libres:                                    # APILAR cogido sobre y
            nuevo = set(hechos) - {('cogido',cogido),('libre',y)} | {('sobre',cogido,y),('libre',cogido),('mano_vacia',)}
            res.append((f"apilar {cogido} sobre {y}", frozenset(nuevo)))
    return res

print("Acciones aplicables ahora mismo desde el estado inicial:")
for nombre, _ in acciones(inicio): print("  -", nombre)


Acciones aplicables ahora mismo desde el estado inicial:
  - coger B de la mesa
  - desapilar C de A


## 3. Planificar = buscar el plan más corto

Una vez tenemos estados y acciones, planificar es **buscar**: desde el estado inicial, aplica acciones
posibles, generando nuevos estados, hasta alcanzar uno que cumpla la meta. Es exactamente la búsqueda
en anchura (BFS) del primer cuaderno de este facsímil, pero en un espacio donde los «nodos» son
situaciones del mundo. BFS nos garantiza el plan **más corto** en número de acciones. Contamos cuántos
estados explora.


In [3]:
def planificar(inicio, meta):
    if meta <= inicio: return [], 1
    frontera = deque([(inicio, [])]); vistos = {inicio}; explorados = 0
    while frontera:
        estado, plan = frontera.popleft(); explorados += 1
        for nombre, nuevo in acciones(estado):
            if nuevo in vistos: continue
            if meta <= nuevo:
                return plan + [nombre], explorados
            vistos.add(nuevo); frontera.append((nuevo, plan + [nombre]))
    return None, explorados

plan, explorados = planificar(inicio, meta)
print(f"Plan encontrado en {len(plan)} pasos (explorando {explorados} estados):\n")
for i, paso in enumerate(plan, 1):
    print(f"  {i}. {paso}")


Plan encontrado en 6 pasos (explorando 16 estados):

  1. desapilar C de A
  2. dejar C en la mesa
  3. coger B de la mesa
  4. apilar B sobre C
  5. coger A de la mesa
  6. apilar A sobre B


**Fíjate en lo que NO ha pasado.** En ningún momento le dijimos «primero quita C de encima de A».
Solo describimos las cuatro acciones con sus reglas, y el planificador **dedujo** que, para poner A
sobre B, antes hay que liberar A, y para eso quitar C. Eso es planificar: razonar la secuencia a partir
del modelo, no ejecutarla de memoria. Comprobemos que el plan, aplicado paso a paso, lleva de verdad a
la meta.


In [4]:
estado = inicio
print("Ejecución del plan, paso a paso:")
dibuja(estado)
for paso in plan:
    estado = dict(acciones(estado))[paso]
    print(f"  tras '{paso}':"); dibuja(estado)
print("\n¿Cumple la meta?:", meta <= estado)


Ejecución del plan, paso a paso:
C/A  B  (sobre la mesa)
  tras 'desapilar C de A':
A  B  (sobre la mesa)
  tras 'dejar C en la mesa':
A  B  C  (sobre la mesa)
  tras 'coger B de la mesa':
A  C  (sobre la mesa)
  tras 'apilar B sobre C':
A  B/C  (sobre la mesa)
  tras 'coger A de la mesa':
B/C  (sobre la mesa)
  tras 'apilar A sobre B':
A/B/C  (sobre la mesa)

¿Cumple la meta?: True


## 4. El problema del tamaño: cuando el espacio explota

La planificación por búsqueda ciega tiene un talón de Aquiles: el espacio de estados **crece de forma
explosiva** con el número de objetos. Cada bloque nuevo multiplica las situaciones posibles. Lo
medimos: planificamos la misma idea (apilar todos los bloques en una torre) con 3, 4 y 5 bloques, y
miramos cuántos estados hay que explorar y cuánto tarda.


In [5]:
def problema_torre(n):
    # n bloques en la mesa; meta: torre B0 sobre B1 sobre ... sobre B(n-1)
    nombres = [f"B{i}" for i in range(n)]
    ini = set()
    for b in nombres: ini |= {('mesa', b), ('libre', b)}
    ini.add(('mano_vacia',))
    objetivo = {('sobre', nombres[i], nombres[i+1]) for i in range(n-1)}
    return frozenset(ini), objetivo

print("bloques | pasos del plan | estados explorados |  tiempo")
for n in [3, 4, 5]:
    ini, obj = problema_torre(n)
    t0 = time.perf_counter()
    pl, ex = planificar(ini, obj)
    dt = time.perf_counter() - t0
    print(f"   {n}    |      {len(pl):>3}       |      {ex:>7,}       | {dt:.3f}s")
print("\nCada bloque dispara el numero de estados: por eso la planificacion real usa HEURISTICAS")
print("(estimar 'cuanto falta'), igual que A* hacia con la busqueda en el primer cuaderno.")


bloques | pasos del plan | estados explorados |  tiempo
   3    |        4       |           13       | 0.000s
   4    |        6       |           80       | 0.001s
   5    |        8       |          639       | 0.006s

Cada bloque dispara el numero de estados: por eso la planificacion real usa HEURISTICAS
(estimar 'cuanto falta'), igual que A* hacia con la busqueda en el primer cuaderno.


**Esa tabla es la razón de ser del capítulo 10.** Con búsqueda ciega, pasar de 3 a 5 bloques
multiplica los estados explorados de forma brutal. Ningún problema real (con decenas de objetos) se
resuelve así. La solución es la misma que en el primer cuaderno: **heurísticas** que estimen cuánto
falta para la meta y guíen la búsqueda (planificación heurística), además de técnicas como traducir el
problema a SAT. La idea de fondo no cambia: planificar es buscar, y buscar bien es buscar con criterio.


## 5. Pruébalo tú

1. **Cambia la meta** a `{('sobre','C','B'),('sobre','B','A')}` (la torre al revés). ¿Cuántos pasos y
   cuántos estados ahora? ¿Es más fácil o más difícil partiendo del mismo inicio?
2. **Añade un bloque D** al estado inicial y mételo en la torre. Observa cómo suben los estados
   explorados: estás tocando la explosión combinatoria con el dedo.
3. **Quita una precondición** (por ejemplo, deja apilar sobre un bloque no libre) y mira cómo el
   planificador encuentra «planes» imposibles. Las precondiciones son las que mantienen la cordura.
4. **Cuenta acciones distintas:** ¿cuántas acciones aplicables hay en el estado inicial? Ese número
   (el *factor de ramificación*) es lo que hace crecer el árbol.


## 6. Errores comunes

- **Olvidar un efecto de borrado.** Si una acción añade «cogido» pero no borra «sobre la mesa», el
  estado queda inconsistente (el bloque está a la vez cogido y en la mesa). Los efectos de borrado son
  tan importantes como los de adición.
- **Confundir el plan con su ejecución.** El planificador *deduce* la secuencia; ejecutarla es otra
  fase. Separar ambas cosas es justo lo que da robustez a un agente.
- **Esperar que la búsqueda ciega escale.** No lo hace. En cuanto hay varios objetos, hacen falta
  heurísticas (capítulo 10).
- **Modelar mal el dominio.** El 90% de los fallos de un planificador son precondiciones o efectos mal
  escritos, no el algoritmo. Modelar bien es el verdadero trabajo.


## 7. Qué te llevas

- **Planificar = modelar acciones** (precondiciones + efectos) y **buscar** la secuencia que lleva del
  estado inicial a la meta. El modelo es declarativo; el plan, emergente.
- Es **la misma búsqueda** del primer cuaderno, ahora sobre estados del mundo en vez de casillas de un
  mapa. La maquinaria se reutiliza.
- Escala mal por fuerza bruta: el espacio explota con los objetos, de ahí las **heurísticas** (capítulo
  10) y, hoy, los agentes LLM que proponen planes y los validan contra un modelo como este.

**Para seguir:** el capítulo 10 mete heurísticas y SAT en la planificación; el facsímil 5 lleva esta
idea a agentes que planifican y actúan con herramientas reales, donde el «plan» se ejecuta paso a paso
y se replanifica si el mundo no responde como se esperaba.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*